##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Zero-Cloud Hybrid RAG with SQLite FTS5, text-embedding-004, and Gemini 2.5 Flash

_Authored by: [Çağrı Giray Keşan](https://github.com/Cagrik34)_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Zero_Cloud_Hybrid_RAG_SQLite_FTS5_Gemini.ipynb)

---

## 📌 Overview

Standard dense-only vector retrieval frequently struggles with **exact-match queries** (e.g. monetary figures, policy limits, contract identifiers, version numbers). Conversely, pure keyword search fails on semantic phrasing.

This recipe implements an embedded, **Zero-Cloud Dual-Engine Hybrid RAG architecture**:
1. **Embedded SQLite FTS5 (BM25):** Sparse full-text keyword indexing using `unicode61` tokenization.
2. **Google `text-embedding-004`:** Dense semantic representation.
3. **Reciprocal Rank Fusion (RRF, $k=60$):** Merges disparate score distributions into a single calibrated ranking.
4. **Grounded Synthesis via the Interactions API:** Streams verified answers with strict source citation anchors (`[1]`, `[2]`).

## 📦 1. Installation & Environment Setup

In [ ]:
%pip install -q -U "google-genai>=2.9.0" numpy

import os
from google import genai

# Set up the Google GenAI client
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
print(f"✅ GenAI Client Initialized: {'Live API Connected' if client else 'Local Fallback Simulation Active'}")

## 🗄️ 2. SQLite Hybrid Store (Dense Vectors + FTS5 BM25 + RRF)

In [ ]:
import sqlite3
import numpy as np
from typing import List, Dict, Any, Tuple

class SQLiteHybridRAGStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS document_chunks (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_file TEXT NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                );
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS document_chunks_fts USING fts5(
                    content,
                    source_file UNINDEXED,
                    tokenize='unicode61'
                );
            """)

    def insert_chunk(self, source_file: str, content: str, embedding: np.ndarray) -> int:
        norm = np.linalg.norm(embedding)
        normalized_vec = (embedding / norm).astype(np.float32) if norm > 0 else embedding.astype(np.float32)
        emb_blob = normalized_vec.tobytes()

        with self.conn:
            cursor = self.conn.cursor()
            cursor.execute(
                "INSERT INTO document_chunks (source_file, content, embedding) VALUES (?, ?, ?)",
                (source_file, content, emb_blob)
            )
            doc_id = cursor.lastrowid
            cursor.execute(
                "INSERT INTO document_chunks_fts (rowid, content, source_file) VALUES (?, ?, ?)",
                (doc_id, content, source_file)
            )
            return doc_id

    def search_dense(self, query_vec: np.ndarray, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        norm = np.linalg.norm(query_vec)
        q_norm = (query_vec / norm).astype(np.float32) if norm > 0 else query_vec.astype(np.float32)

        cursor = self.conn.cursor()
        cursor.execute("SELECT id, source_file, content, embedding FROM document_chunks")
        results = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            score = float(np.dot(q_norm, doc_vec))
            results.append((doc_id, src, content, score))
        
        return sorted(results, key=lambda x: x[3], reverse=True)[:top_k]

    def search_sparse_bm25(self, query_text: str, top_k: int = 5) -> List[Tuple[int, str, str, float]]:
        tokens = [t.replace("'", "").replace('"', '') for t in query_text.split() if t.strip()]
        if not tokens:
            return []
        sanitized_query = " OR ".join([f'"{t}"' for t in tokens])
        
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT rowid, source_file, content, rank
            FROM document_chunks_fts
            WHERE document_chunks_fts MATCH ?
            ORDER BY rank
            LIMIT ?
        """, (sanitized_query, top_k))
        
        hits = []
        for doc_id, src, content, rank in cursor.fetchall():
            bm25_score = 1.0 / (1.0 + abs(float(rank)))
            hits.append((doc_id, src, content, bm25_score))
        return hits

    def hybrid_search(self, query_text: str, query_vec: np.ndarray, top_k: int = 3, rrf_k: int = 60) -> List[Dict[str, Any]]:
        dense_hits = self.search_dense(query_vec, top_k=top_k * 2)
        sparse_hits = self.search_sparse_bm25(query_text, top_k=top_k * 2)
        chunk_map = {}
        fused_scores = {}

        # Fuse dense ranks using stable doc_id
        for rank, (doc_id, src, content, _) in enumerate(dense_hits, start=1):
            chunk_map[doc_id] = (src, content, "dense")
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))

        # Fuse sparse BM25 ranks using aligned doc_id
        for rank, (doc_id, src, content, _) in enumerate(sparse_hits, start=1):
            if doc_id not in chunk_map:
                chunk_map[doc_id] = (src, content, "bm25")
            else:
                src, content, _ = chunk_map[doc_id]
                chunk_map[doc_id] = (src, content, "hybrid")
            fused_scores[doc_id] = fused_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank))

        sorted_doc_ids = sorted(fused_scores.keys(), key=lambda d_id: fused_scores[d_id], reverse=True)[:top_k]
        output = []
        for idx, doc_id in enumerate(sorted_doc_ids, start=1):
            src, content, match_type = chunk_map[doc_id]
            output.append({
                "citation_index": idx,
                "doc_id": doc_id,
                "source_file": src,
                "content": content,
                "rrf_score": round(fused_scores[doc_id], 4),
                "match_type": match_type
            })
        return output

print("✅ SQLiteHybridRAGStore defined successfully with aligned doc_id indexing.")

## 📄 3. Document Ingestion with text-embedding-004

In [ ]:
store = SQLiteHybridRAGStore()

documents = [
    ("q3_financial_report.pdf", "Core infrastructure engineering Q3 total budget was finalized at 2,340,000 TL with 15 developers."),
    ("architecture_specs.md", "Zenith AI leverages local on-device small language models for zero-cloud edge inference."),
    ("hr_policy_2026.docx", "Quarterly remote work equipment allowance is strictly capped at 15,000 TL per developer."),
    ("cluster_ops.md", "Kubernetes horizontal pod autoscaler scales pods when memory utilization exceeds 80% for 5 minutes.")
]

print("Ingesting corpus into embedded SQLite...")
for src, content in documents:
    if client:
        res = client.models.embed_content(
            model="text-embedding-004",
            contents=content
        )
        emb = np.array(res.embeddings[0].values, dtype=np.float32)
    else:
        # Deterministic simulation vector for local / CI environments
        np.random.seed(abs(hash(content)) % 10000)
        emb = np.random.randn(768).astype(np.float32)
    
    store.insert_chunk(src, content, emb)

print(f"✅ Successfully ingested {len(documents)} document chunks into SQLite.")

## 🔍 4. Hybrid Query Execution & Rank Fusion ($k=60$)

In [ ]:
query = "What is the quarterly remote work allowance limit in TL?"
print(f"Query: '{query}'\n")

if client:
    q_res = client.models.embed_content(
        model="text-embedding-004",
        contents=query
    )
    query_vec = np.array(q_res.embeddings[0].values, dtype=np.float32)
else:
    np.random.seed(abs(hash(query)) % 10000)
    query_vec = np.random.randn(768).astype(np.float32)

results = store.hybrid_search(query, query_vec, top_k=2)

print("--- Retrieved Citations (SQLite FTS5 + text-embedding-004 via RRF k=60) ---")
for r in results:
    print(f"[{r['citation_index']}] Source: {r['source_file']} | Match: {r['match_type'].upper()} | RRF Score: {r['rrf_score']}")
    print(f"    Content: {r['content']}\n")

## 🤖 5. Grounded Answer Synthesis with Gemini (Interactions API)

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-3.7-flash"] {"allow-input": true, "isTemplate": true}

context_str = "\n\n".join([f"[{r['citation_index']}] (Source: {r['source_file']}) {r['content']}" for r in results])

system_instruction = (
    "You are a precise corporate assistant. Answer the user prompt strictly based on the provided context.\n"
    "For every factual statement or numerical figure, append the exact source citation index in brackets ([1], [2]).\n\n"
    f"Context:\n{context_str}"
)

print("--- Grounded Model Response ---\n")
if client:
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=query,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1
        )
    )
    print(interaction.steps[-1].content[0].text)
else:
    print("According to the HR policy documentation [1], the quarterly remote work equipment allowance is strictly capped at 15,000 TL per developer.")